# Fundamentals of Machine Learning — Exercise 10
## Neural Networks from a Perceptron to a Small MLP

A neural network is not a mysterious object that becomes understandable only after importing a deep-learning framework. In this exercise we first build the important pieces ourselves and only then compare them with a library implementation.

The classroom workflow remains:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

After completing this exercise, you should be able to:

- explain a perceptron as a weighted sum followed by an activation,
- recognise which binary patterns are and are not linearly separable,
- track the shapes of matrices through a two-layer neural network,
- compute a forward pass using NumPy,
- explain the roles of hidden units and nonlinear activations,
- connect binary cross-entropy to probabilistic classification,
- interpret backpropagation as repeated application of the chain rule,
- verify an analytical gradient numerically,
- train a small MLP using prepared NumPy code,
- compare a NumPy MLP with logistic regression and `MLPClassifier`,
- diagnose underfitting and overfitting using validation evidence,
- use AI to question a debugging argument rather than generate the network for you.

**Recommended duration:** 110–120 minutes.  
The final debugging/model-selection task may be completed as homework.

## How to work in this exercise

Most code is prepared. Your task is to:

1. **predict** matrix shapes or model behaviour before running a cell,
2. **trace** one computation instead of treating the network as a black box,
3. **change** only clearly marked settings,
4. **compare** simple and nonlinear models,
5. **verify** at least one gradient or modelling claim independently.

A network with more layers is not automatically a better model.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 1. A perceptron is a linear decision unit

For an input vector $x$, a perceptron first computes

$$z = w^T x + b$$

and then applies a threshold.

We begin with two tiny logical problems: **AND** and **XOR**.

In [ ]:
X_logic = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y_and = np.array([0,0,0,1])
y_xor = np.array([0,1,1,0])

logic = pd.DataFrame(X_logic, columns=['x1','x2'])
logic['AND'] = y_and
logic['XOR'] = y_xor
logic

### Predict before training

1. Can one straight line separate the AND classes?
2. Can one straight line separate the XOR classes?
3. If your answer differs, sketch the reason using the four points.

In [ ]:
def train_perceptron(X, y, learning_rate=0.2, epochs=25):
    w = np.zeros(X.shape[1])
    b = 0.0
    mistakes = []
    for _ in range(epochs):
        epoch_mistakes = 0
        for xi, yi in zip(X, y):
            pred = int(np.dot(xi, w) + b >= 0)
            update = learning_rate * (yi - pred)
            w += update * xi
            b += update
            epoch_mistakes += int(pred != yi)
        mistakes.append(epoch_mistakes)
    return w, b, mistakes

and_w, and_b, and_mistakes = train_perceptron(X_logic, y_and)
xor_w, xor_b, xor_mistakes = train_perceptron(X_logic, y_xor)

pd.DataFrame({
    'epoch': np.arange(1, len(and_mistakes)+1),
    'AND mistakes': and_mistakes,
    'XOR mistakes': xor_mistakes,
}).head(10)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(and_mistakes, marker='o', label='AND')
plt.plot(xor_mistakes, marker='o', label='XOR')
plt.xlabel('Epoch')
plt.ylabel('Misclassified points')
plt.title('Perceptron training')
plt.legend()
plt.show()

### Activity 1 — Interpret the failure

Complete:

> The perceptron can solve __________ because __________.
>
> It cannot solve __________ because __________.
>
> Repeating training for more epochs does not fix the second problem because __________.

## 2. Why a hidden layer helps

A multilayer perceptron (MLP) combines several linear transformations with **nonlinear activation functions**. The nonlinear step is essential; stacking only linear transformations would still be equivalent to one linear transformation.

We will use `tanh` in the hidden layer and `sigmoid` at the binary output.

In [ ]:
z = np.linspace(-4, 4, 300)
sigmoid_curve = 1 / (1 + np.exp(-z))
tanh_curve = np.tanh(z)
relu_curve = np.maximum(0, z)

plt.figure(figsize=(9,5))
plt.plot(z, sigmoid_curve, label='sigmoid')
plt.plot(z, tanh_curve, label='tanh')
plt.plot(z, relu_curve, label='ReLU')
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel('z')
plt.ylabel('activation(z)')
plt.title('Common activation functions')
plt.legend()
plt.show()

### Activity 2 — Connect shape to behaviour

For each activation, state:

- its approximate output range,
- whether negative inputs can produce negative outputs,
- one consequence of using it in a hidden layer.

Why would replacing `tanh` by the identity function remove the key benefit of the hidden layer?

## 3. Trace one forward pass by matrix shape

Our microscopic network has:

- 2 input features,
- 3 hidden units,
- 1 output unit.

For a batch of 4 observations, predict the shape of each object **before running the cell**.

| Object | Predicted shape |
|---|---|
| `X` | |
| `W1` | |
| `b1` | |
| `Z1 = X @ W1 + b1` | |
| `A1` | |
| `W2` | |
| `Z2 = A1 @ W2 + b2` | |
| `y_hat` | |

In [ ]:
X = X_logic.copy()
W1 = np.array([[ 1.0, -1.0,  0.5],
               [-0.5,  1.0,  1.0]])
b1 = np.array([[0.0, 0.2, -0.1]])
W2 = np.array([[1.0], [-1.2], [0.8]])
b2 = np.array([[0.1]])

Z1 = X @ W1 + b1
A1 = np.tanh(Z1)
Z2 = A1 @ W2 + b2
y_hat = 1 / (1 + np.exp(-Z2))

shape_table = pd.Series({
    'X': X.shape, 'W1': W1.shape, 'b1': b1.shape,
    'Z1': Z1.shape, 'A1': A1.shape, 'W2': W2.shape,
    'Z2': Z2.shape, 'y_hat': y_hat.shape,
})
shape_table

In [ ]:
pd.DataFrame({
    'x1': X[:,0], 'x2': X[:,1],
    'predicted probability': y_hat.ravel(),
    'XOR target': y_xor,
}).round(3)

### Stop and explain

1. Why does `W1` have shape `(2, 3)`?
2. Why does `W2` have shape `(3, 1)`?
3. What information does one row of `A1` contain?
4. Why is the output a probability rather than a class label?

## 4. Debug a deliberate matrix-shape error

The following weight matrix is wrong for our architecture. The cell catches the error so that the notebook continues running.

In [ ]:
BROKEN_W2 = np.array([[1.0], [-1.0]])  # wrong number of rows

try:
    _ = A1 @ BROKEN_W2
except ValueError as error:
    print('Caught error:')
    print(error)
    print('\nA1 shape:', A1.shape)
    print('BROKEN_W2 shape:', BROKEN_W2.shape)

### Activity 3 — Diagnose before fixing

Do not ask AI yet.

1. Which dimensions must match in `A1 @ W2`?
2. What shape must `W2` have?
3. Which dimension corresponds to hidden units?
4. Write the corrected shape before looking back at the architecture.

## 5. Binary cross-entropy and backpropagation

For binary classification we use

$$L=-\frac{1}{n}\sum_i[y_i\log(\hat y_i)+(1-y_i)\log(1-\hat y_i)].$$

Backpropagation is the chain rule organised efficiently from the output layer back toward the input.

For sigmoid output + binary cross-entropy, the output-layer derivative simplifies to

$$dZ_2 = \hat y - y.$$

The remaining gradients follow by matrix multiplication and element-wise derivatives.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1 / (1 + np.exp(-z))

def bce(y, p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    Z2 = A1 @ W2 + b2
    P = sigmoid(Z2)
    return Z1, A1, Z2, P

def gradients(X, y, W1, b1, W2, b2):
    y = y.reshape(-1,1)
    Z1, A1, Z2, P = forward(X, W1, b1, W2, b2)
    n = len(X)
    dZ2 = (P - y) / n
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (1 - np.tanh(Z1)**2)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0, keepdims=True)
    return dW1, db1, dW2, db2

gW1, gb1, gW2, gb2 = gradients(X_logic, y_xor, W1, b1, W2, b2)
pd.Series({'dW1': gW1.shape, 'db1': gb1.shape, 'dW2': gW2.shape, 'db2': gb2.shape})

### Predict the gradient shapes

Why must every gradient have the same shape as the parameter it updates?

Explain the meaning of `A1.T @ dZ2` in words rather than code.

### Numerical gradient check

We verify one analytical derivative by changing one weight by a very small amount. If backpropagation is correct, the analytical and numerical gradients should be very similar.

In [ ]:
def loss_with_params(X, y, W1, b1, W2, b2):
    P = forward(X, W1, b1, W2, b2)[-1]
    return bce(y.reshape(-1,1), P)

eps = 1e-5
W2_plus = W2.copy(); W2_plus[0,0] += eps
W2_minus = W2.copy(); W2_minus[0,0] -= eps

numerical = (
    loss_with_params(X_logic, y_xor, W1, b1, W2_plus, b2)
    - loss_with_params(X_logic, y_xor, W1, b1, W2_minus, b2)
) / (2*eps)
analytical = gW2[0,0]

pd.Series({
    'analytical gradient': analytical,
    'numerical gradient': numerical,
    'absolute difference': abs(analytical-numerical),
})

### Verification checkpoint

1. Are the two gradients close enough to support the implementation?
2. What would a large discrepancy suggest?
3. Why is gradient checking useful for learning/debugging but expensive for full neural-network training?

## 6. Train a small NumPy MLP on XOR

The training loop is supplied. Focus on how loss changes and whether the nonlinear model solves the pattern that the perceptron could not.

In [ ]:
def train_numpy_mlp(X, y, hidden=4, learning_rate=0.3, epochs=2500, seed=42):
    local_rng = np.random.default_rng(seed)
    W1 = local_rng.normal(0, 0.6, size=(X.shape[1], hidden))
    b1 = np.zeros((1, hidden))
    W2 = local_rng.normal(0, 0.6, size=(hidden, 1))
    b2 = np.zeros((1,1))
    losses = []
    for epoch in range(epochs):
        P = forward(X, W1, b1, W2, b2)[-1]
        losses.append(bce(y.reshape(-1,1), P))
        dW1, db1, dW2, db2 = gradients(X, y, W1, b1, W2, b2)
        W1 -= learning_rate*dW1
        b1 -= learning_rate*db1
        W2 -= learning_rate*dW2
        b2 -= learning_rate*db2
    return (W1,b1,W2,b2), np.array(losses)

xor_params, xor_losses = train_numpy_mlp(X_logic, y_xor, hidden=4, learning_rate=0.5, epochs=3000)
xor_probs = forward(X_logic, *xor_params)[-1].ravel()
xor_pred = (xor_probs >= 0.5).astype(int)

pd.DataFrame({'target':y_xor, 'probability':xor_probs, 'prediction':xor_pred}).round(3)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(xor_losses)
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.title('NumPy MLP learning on XOR')
plt.show()

### Activity 4 — What changed relative to the perceptron?

1. Did the MLP solve XOR?
2. What representational ingredient did the MLP add?
3. Does a low training loss alone prove that the architecture will generalise to new data?

## 7. Real nonlinear data: two moons

We now compare:

- logistic regression,
- our NumPy MLP,
- scikit-learn `MLPClassifier`.

The target is deliberately nonlinear.

In [ ]:
X_moons, y_moons = make_moons(n_samples=700, noise=0.22, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.30, stratify=y_moons, random_state=RANDOM_STATE
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

plt.figure(figsize=(7,5))
plt.scatter(X_train_s[:,0], X_train_s[:,1], c=y_train, cmap='coolwarm', s=28, alpha=0.75)
plt.xlabel('Scaled feature 1'); plt.ylabel('Scaled feature 2'); plt.title('Training data: two moons')
plt.show()

### Predict before fitting

Rank the three models by expected ability to follow the curved boundary. Give one reason for each position.

In [ ]:
logistic = LogisticRegression().fit(X_train_s, y_train)

numpy_params, numpy_losses = train_numpy_mlp(
    X_train_s, y_train, hidden=8, learning_rate=0.08, epochs=1800, seed=3
)

sk_mlp = MLPClassifier(
    hidden_layer_sizes=(8,), activation='tanh', max_iter=1800,
    random_state=3, learning_rate_init=0.01
).fit(X_train_s, y_train)

predictions = {
    'Logistic regression': logistic.predict_proba(X_test_s)[:,1],
    'NumPy MLP': forward(X_test_s, *numpy_params)[-1].ravel(),
    'sklearn MLPClassifier': sk_mlp.predict_proba(X_test_s)[:,1],
}

rows=[]
for name,p in predictions.items():
    rows.append({
        'model':name,
        'accuracy':accuracy_score(y_test, p>=0.5),
        'log loss':log_loss(y_test, p),
    })
pd.DataFrame(rows).round(3)

In [ ]:
def plot_boundary(ax, model_name, probability_function):
    x0_min, x0_max = X_train_s[:,0].min()-0.7, X_train_s[:,0].max()+0.7
    x1_min, x1_max = X_train_s[:,1].min()-0.7, X_train_s[:,1].max()+0.7
    xx, yy = np.meshgrid(np.linspace(x0_min,x0_max,180), np.linspace(x1_min,x1_max,180))
    grid = np.c_[xx.ravel(), yy.ravel()]
    pp = probability_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, pp, levels=np.linspace(0,1,12), alpha=0.35, cmap='coolwarm')
    ax.contour(xx, yy, pp, levels=[0.5], linewidths=2)
    ax.scatter(X_test_s[:,0], X_test_s[:,1], c=y_test, cmap='coolwarm', s=20, edgecolor='k', linewidth=0.2)
    ax.set_title(model_name)

fig, axes = plt.subplots(1,3,figsize=(15,4.5))
plot_boundary(axes[0], 'Logistic regression', lambda X: logistic.predict_proba(X)[:,1])
plot_boundary(axes[1], 'NumPy MLP', lambda X: forward(X, *numpy_params)[-1].ravel())
plot_boundary(axes[2], 'sklearn MLP', lambda X: sk_mlp.predict_proba(X)[:,1])
plt.tight_layout(); plt.show()

### Activity 5 — Compare behaviour, not just accuracy

1. Which boundary is constrained to be linear?
2. Which two models can represent a curved boundary?
3. Are the two MLP boundaries identical? Why should they not have to be?
4. Which evidence would you use before claiming that the more flexible boundary is better?

## 8. Capacity is a modelling choice

We compare hidden-layer widths. A larger network has more parameters and therefore more capacity, but capacity must be justified by validation evidence.

In [ ]:
widths = [2, 8, 32]
capacity_rows=[]
for width in widths:
    model = MLPClassifier(
        hidden_layer_sizes=(width,), activation='tanh',
        max_iter=1800, random_state=5, learning_rate_init=0.01
    ).fit(X_train_s, y_train)
    capacity_rows.append({
        'hidden units': width,
        'parameters approx.': 2*width + width + width + 1,
        'train accuracy': model.score(X_train_s, y_train),
        'test accuracy': model.score(X_test_s, y_test),
        'test log loss': log_loss(y_test, model.predict_proba(X_test_s)[:,1]),
    })
pd.DataFrame(capacity_rows).round(3)

### Activity 6 — Diagnose capacity

Which width would you choose from this experiment and why? Mention:

- training evidence,
- test evidence (for this teaching experiment),
- parameter count,
- one reason not to generalise the conclusion to every dataset.

## 9. Guided AI study activity — debug your reasoning

First write your own explanation of the matrix-shape error from Section 4 and your model choice from Section 8.

Then use this prompt:

```text
Act as a neural-network tutor. Do not write code and do not give me the corrected matrix directly.
Ask me exactly three short questions, one at a time:
1. What do the rows and columns of each matrix represent?
2. Which dimensions must match during matrix multiplication?
3. What independent evidence supports my model-capacity choice?
Wait for my answer after each question.
```

Close the chat and record only:

**One useful AI question:**  

**One statement I corrected:**  

**Why my correction is better:**

## 10. Assessed task — Network debugging and model selection (2 points)

Change **two** clearly marked settings:

- `TASK_HIDDEN_UNITS`,
- `TASK_LEARNING_RATE`.

Compare at least three configurations of the NumPy MLP on the two-moons data. Submit:

1. your predicted effect of each change,
2. final training loss,
3. test accuracy and log loss,
4. one matrix-shape statement that you can defend,
5. one comparison with logistic regression or `MLPClassifier`,
6. your recommended configuration,
7. one limitation.

The goal is not the highest possible score. The goal is a defensible explanation.

In [ ]:
TASK_HIDDEN_UNITS = 8
TASK_LEARNING_RATE = 0.08
TASK_EPOCHS = 1800

params_task, loss_task = train_numpy_mlp(
    X_train_s, y_train,
    hidden=TASK_HIDDEN_UNITS,
    learning_rate=TASK_LEARNING_RATE,
    epochs=TASK_EPOCHS,
    seed=9,
)
p_task = forward(X_test_s, *params_task)[-1].ravel()

pd.Series({
    'hidden units': TASK_HIDDEN_UNITS,
    'learning rate': TASK_LEARNING_RATE,
    'final training loss': loss_task[-1],
    'test accuracy': accuracy_score(y_test, p_task>=0.5),
    'test log loss': log_loss(y_test, p_task),
}).round(4)

### Your network-selection note

**Prediction:**  

**Configurations compared:**  

**Evidence:**  

**Matrix-shape rule:**  

**Comparison with a library/baseline:**  

**Recommendation:**  

**Limitation:**  

**Optional verified AI note:**

# End-of-lab self-check

- [ ] I can explain why XOR defeats a single perceptron.
- [ ] I can state the shapes in a two-layer network.
- [ ] I can explain why hidden activations must be nonlinear.
- [ ] I can describe a forward pass in words.
- [ ] I can explain the purpose of binary cross-entropy.
- [ ] I can connect backpropagation to the chain rule.
- [ ] I can explain what a numerical gradient check verifies.
- [ ] I can compare a NumPy MLP with a library model.
- [ ] I can justify network capacity using evidence rather than “bigger is better”.
- [ ] I can use AI questions to reconstruct an explanation without copying the solution.